# Upload de Stems para Cloudflare R2

**Instrucoes:** Clique em **Executar tudo** no menu acima (ou `Ctrl+F9`).

O unico passo manual e fazer login no Google quando aparecer a tela de autorizacao.

In [ ]:
# === CELULA 1: Instalar dependencias + Configuracao ===
!pip install boto3 google-auth google-auth-oauthlib google-api-python-client -q

import boto3
import os
import json
import time
import zipfile
import shutil
import unicodedata
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

# === Credenciais R2 (ja preenchidas) ===
R2_ENDPOINT = "https://cdc3f4352ba3c01fe88a9a77a92d2aa0.r2.cloudflarestorage.com"
R2_ACCESS_KEY_ID = "3737acd0ce1cd77d61a7917c0035e628"
R2_SECRET_ACCESS_KEY = "fb8ced7d8cf81825bd222797cee9c26ffa1b332e27fe6ab6b1aa8f12ed769590"
R2_BUCKET = "tom"
R2_PUBLIC_URL = "https://pub-81339f96e6c746e8ac25fbe60e95563c.r2.dev"

SHARED_FOLDER_ID = "1cVaUf3nolyYwDwf0ynPUZuPrazEh0QZy"

GENRE_MAP = {
    "01. ATUALIZAÇÕES 2017-2026": "atualizacoes",
    "02. FORRÓ DAS ANTIGAS": "forro",
    "03. PAGODES": "pagode",
    "04. SERTANEJO": "sertanejo",
    "05. GOSPEL": "gospel",
    "06. ROCK POP MPB BREGA": "rock-pop-mpb",
    "07. AXÉS, CARNAVAL E PAGODE BAIANO": "axe-carnaval",
    "08. ABERTURAS DE SHOW": "aberturas",
    "09. PLAYBACKS FECHADOS": "playbacks",
    "10. PASTA DE SHOWS MONTADOS MULTIPISTAS REAPER": "shows-multipistas",
    "11. PASTA DE SHOWS MONTADOS PLAYBACKS FECHADOS": "shows-playbacks",
}

AUDIO_EXTENSIONS = {".mp3", ".wav", ".ogg", ".m4a"}
CONTENT_TYPES = {".mp3": "audio/mpeg", ".wav": "audio/wav", ".ogg": "audio/ogg", ".m4a": "audio/mp4"}

def slugify(text):
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")

print("Configuracao OK!")

In [ ]:
# === CELULA 2: Autenticar no Google Drive + Conectar R2 ===
# >> AQUI VAI APARECER UMA TELA DE LOGIN — FACA LOGIN NA SUA CONTA GOOGLE <<

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
drive_service = build("drive", "v3")

# Testar acesso ao Drive
results = drive_service.files().list(
    q=f"'{SHARED_FOLDER_ID}' in parents and mimeType='application/vnd.google-apps.folder'",
    fields="files(id, name)",
    supportsAllDrives=True,
    includeItemsFromAllDrives=True,
    orderBy="name",
).execute()

drive_folders = results.get("files", [])
print(f"Autenticado! Encontradas {len(drive_folders)} pastas de genero:")
for f in drive_folders:
    print(f"  {f['name']}")

# Conectar ao R2
s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
try:
    s3.head_bucket(Bucket=R2_BUCKET)
    print(f"\nConectado ao R2 (bucket: {R2_BUCKET})")
except Exception as e:
    print(f"ERRO ao acessar R2: {e}")
    raise

In [ ]:
# === CELULA 3: Detectar progresso automaticamente do R2 ===
# Nao precisa mais fazer upload de arquivo — escaneia o R2 direto!

print("Escaneando arquivos ja existentes no R2...")

paginator_token = None
completed_set = set()

while True:
    kwargs = {"Bucket": R2_BUCKET, "Prefix": "stems/", "MaxKeys": 1000}
    if paginator_token:
        kwargs["ContinuationToken"] = paginator_token
    resp = s3.list_objects_v2(**kwargs)
    for obj in resp.get("Contents", []):
        parts = obj["Key"].split("/")
        if len(parts) >= 4:
            completed_set.add(f"{parts[1]}/{parts[2]}")
    paginator_token = resp.get("NextContinuationToken")
    if not paginator_token:
        break

print(f"Musicas ja no R2: {len(completed_set)}")
catalog = []  # sera reconstruido no final

In [ ]:
# === CELULA 4: Escanear Drive + Upload só do que falta ===

# --- Funcoes auxiliares ---
def list_files_in_folder(folder_id, mime_filter=None):
    all_files = []
    page_token = None
    q = f"'{folder_id}' in parents and trashed=false"
    if mime_filter:
        q += f" and mimeType='{mime_filter}'"
    while True:
        results = drive_service.files().list(
            q=q, fields="nextPageToken, files(id, name, size, mimeType)",
            supportsAllDrives=True, includeItemsFromAllDrives=True,
            pageSize=1000, pageToken=page_token,
        ).execute()
        all_files.extend(results.get("files", []))
        page_token = results.get("nextPageToken")
        if not page_token:
            break
    return all_files

def download_zip_from_drive(file_id, dest_path):
    request = drive_service.files().get_media(fileId=file_id)
    with open(dest_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return dest_path

def find_audio_files(directory):
    audio_files = []
    for root, _, files in os.walk(directory):
        for fname in files:
            if fname.startswith(".") or "__MACOSX" in root:
                continue
            ext = os.path.splitext(fname)[1].lower()
            if ext in AUDIO_EXTENSIONS:
                audio_files.append(os.path.join(root, fname))
    return sorted(audio_files)

def upload_stem_to_r2(local_path, r2_key):
    ext = os.path.splitext(local_path)[1].lower()
    content_type = CONTENT_TYPES.get(ext, "application/octet-stream")
    file_size = os.path.getsize(local_path)
    with open(local_path, "rb") as f:
        s3.put_object(Bucket=R2_BUCKET, Key=r2_key, Body=f,
                      ContentType=content_type,
                      CacheControl="public, max-age=31536000, immutable")
    return file_size

def process_one_zip(zip_info):
    genre_slug = zip_info["genre_slug"]
    song_slug = zip_info["song_slug"]
    file_id = zip_info["file_id"]
    tmp_zip = f"/tmp/stem_{file_id}.zip"
    tmp_dir = f"/tmp/stem_{file_id}"
    try:
        download_zip_from_drive(file_id, tmp_zip)
        os.makedirs(tmp_dir, exist_ok=True)
        with zipfile.ZipFile(tmp_zip, "r") as zf:
            zf.extractall(tmp_dir)
        audio_files = find_audio_files(tmp_dir)
        if not audio_files:
            return None
        stems = []
        def upload_one(audio_path):
            fname = os.path.basename(audio_path)
            stem_name = os.path.splitext(fname)[0]
            ext = os.path.splitext(fname)[1].lower()
            stem_slug = slugify(stem_name)
            r2_key = f"stems/{genre_slug}/{song_slug}/{stem_slug}{ext}"
            size = upload_stem_to_r2(audio_path, r2_key)
            return {"name": stem_name, "slug": stem_slug, "key": r2_key,
                    "url": f"{R2_PUBLIC_URL}/{r2_key}", "format": ext.lstrip("."), "size": size}
        with ThreadPoolExecutor(max_workers=5) as executor:
            futures = {executor.submit(upload_one, ap): ap for ap in audio_files}
            for future in as_completed(futures):
                try:
                    stems.append(future.result())
                except Exception as e:
                    print(f"    Erro stem: {e}")
        return {"name": zip_info["song_name"], "slug": song_slug,
                "genre": zip_info["genre_name"], "genreSlug": genre_slug,
                "stems": sorted(stems, key=lambda s: s["name"]), "stemCount": len(stems)}
    except Exception as e:
        print(f"    ERRO: {e}")
        return None
    finally:
        if os.path.exists(tmp_zip): os.remove(tmp_zip)
        if os.path.exists(tmp_dir): shutil.rmtree(tmp_dir, ignore_errors=True)

def save_progress():
    with open("/tmp/upload-progress.json", "w") as f:
        json.dump({"completed": list(completed_set), "catalog": catalog}, f, ensure_ascii=False)
    with open("/tmp/catalog.json", "w") as f:
        json.dump(catalog, f, ensure_ascii=False, indent=2)

# --- Escanear ZIPs ---
print("\nEscaneando pastas no Drive...")
zips_to_process = []
skipped = 0

for folder in drive_folders:
    genre_name = folder["name"]
    genre_slug = GENRE_MAP.get(genre_name)
    if not genre_slug:
        continue
    print(f"  {genre_name}...", end=" ")
    all_in_folder = list_files_in_folder(folder["id"])
    zip_files = [f for f in all_in_folder if f["name"].lower().endswith(".zip")]
    new = 0
    for zf in zip_files:
        song_name = zf["name"].rsplit(".", 1)[0]
        song_slug = slugify(song_name)
        key = f"{genre_slug}/{song_slug}"
        if key in completed_set:
            skipped += 1
        else:
            zips_to_process.append({"genre_name": genre_name, "genre_slug": genre_slug,
                                    "file_id": zf["id"], "file_name": zf["name"],
                                    "song_name": song_name, "song_slug": song_slug})
            new += 1
    print(f"{new} novos, {skipped} ja feitos")
    time.sleep(0.2)

print(f"\nTotal para processar: {len(zips_to_process)}")
print(f"Ja no R2: {skipped}")

# --- Executar upload ---
if len(zips_to_process) == 0:
    print("\nTudo ja foi enviado!")
else:
    print(f"\nIniciando upload de {len(zips_to_process)} musicas...\n")
    start_time = time.time()
    errors = 0
    done = 0
    total = len(zips_to_process)

    for i, zip_info in enumerate(zips_to_process):
        idx = i + 1
        elapsed = time.time() - start_time
        rate = idx / (elapsed / 60) if elapsed > 60 else 0
        short = zip_info["song_name"][:45]
        print(f"[{idx}/{total}] {short}...", end=" ")

        result = process_one_zip(zip_info)
        if result:
            key = f"{zip_info['genre_slug']}/{zip_info['song_slug']}"
            completed_set.add(key)
            catalog.append(result)
            done += 1
            print(f"OK ({result['stemCount']} stems) [{rate:.1f}/min]")
        else:
            errors += 1
            print("FALHOU")

        if idx % 10 == 0:
            save_progress()

    save_progress()
    mins = (time.time() - start_time) / 60
    print(f"\n{'='*50}")
    print(f"CONCLUIDO em {mins:.1f} min!")
    print(f"  Enviadas: {done}")
    print(f"  Erros: {errors}")
    print(f"  Total no catalogo: {len(catalog)}")

In [ ]:
# === CELULA 5: Baixar resultados ===
from google.colab import files as colab_files

if os.path.exists("/tmp/catalog.json"):
    print(f"catalog.json: {os.path.getsize('/tmp/catalog.json') / 1024:.1f} KB")
    colab_files.download("/tmp/catalog.json")

if os.path.exists("/tmp/upload-progress.json"):
    print(f"upload-progress.json: {os.path.getsize('/tmp/upload-progress.json') / 1024:.1f} KB")
    colab_files.download("/tmp/upload-progress.json")

print("\nPronto! Guarde o upload-progress.json para retomar depois.")